# ARC-AGI-3 Solver — Qwen3.8-27B-FP8 — 25-Game P1 Public Eval

Public/offline evaluation is overridden to the same 25 public games × 1 pass shape. Competition reruns still use the live private game list from the Kaggle gateway.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# Qwen3.8 / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "sergueimakarov/arc3-atlas-src",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
KERNEL_SOURCES: list[str] = []

# New private Kaggle Model (Version 1).
QWEN_MODEL_OWNER = "foysalemonshanto"
QWEN_MODEL_SLUG = "qwen3-8-27b-fp8-repacked-v1"
QWEN_MODEL_REF = f"{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}"
QWEN_MODEL_VARIATION = "hf-fp8"
QWEN_MODEL_VERSION = "1"
QWEN_SERVED_MODEL_NAME = "Qwen/Qwen3.8-27B-FP8"
QWEN_MODEL_PATH = Path(
    f"/kaggle/input/models/{QWEN_MODEL_OWNER}/{QWEN_MODEL_SLUG}/"
    f"pytorch/{QWEN_MODEL_VARIATION}/{QWEN_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the Qwen3.8 Kaggle Model before any expensive setup work starts.
if not QWEN_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "Qwen3.8 Kaggle Model is not attached.\n"
        f"Expected path:\n{QWEN_MODEL_PATH}\n\n"
        "Attach: Qwen3.8 27B FP8 Repacked → PyTorch → hf-fp8 → Version 1"
    )

_required_qwen_files = [
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "outside.safetensors",
    "mtp.safetensors",
    "chat_template.jinja",
]
_missing_qwen_files = [
    name for name in _required_qwen_files if not (QWEN_MODEL_PATH / name).is_file()
]
if _missing_qwen_files:
    raise FileNotFoundError(
        "Qwen3.8 mount is incomplete; missing: " + ", ".join(_missing_qwen_files)
    )

_qwen_layer_shards = sorted(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))
_qwen_safetensors = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
if len(_qwen_layer_shards) != 16 or len(_qwen_safetensors) != 18:
    raise RuntimeError(
        "Unexpected Qwen3.8 checkpoint layout: "
        f"{len(_qwen_layer_shards)} layer shards, "
        f"{len(_qwen_safetensors)} safetensors files."
    )

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[QWEN_MODEL_REF] = str(QWEN_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_QWEN_MODEL_REF": QWEN_MODEL_REF,
    "TAAF_QWEN_MODEL_PATH": str(QWEN_MODEL_PATH),
    "TAAF_QWEN_SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\n✅ Qwen3.8 input configuration ready")
print(f"Model ref:       {QWEN_MODEL_REF}")
print(f"Physical path:   {QWEN_MODEL_PATH}")
print(f"Served model:    {QWEN_SERVED_MODEL_NAME}")
print(f"Safetensors:     {len(_qwen_safetensors)}")
print(f"Layer shards:    {len(_qwen_layer_shards)}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== Qwen3.8 Kaggle Model ===")
print(QWEN_MODEL_PATH)
print("Exists:", QWEN_MODEL_PATH.exists())
print("Safetensors:", len(list(QWEN_MODEL_PATH.glob("*.safetensors"))))
print(
    "Repacked layer shards:",
    len(list(QWEN_MODEL_PATH.glob("model-layers-*.safetensors"))),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


def _patch_qwen38_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    Qwen3.8 Kaggle Model. This avoids copying/forking the large bundled setup
    script and keeps the wheelhouse/GPU/vLLM behavior from the source bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
    }

    replacements = {
        "MODEL_OWNER": QWEN_MODEL_OWNER,
        "MODEL_SLUG": QWEN_MODEL_SLUG,
        "SERVED_MODEL_NAME": QWEN_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for Qwen3.8. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: Qwen3.8 setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_qwen38_setup_commands(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use Qwen3.8.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != QWEN_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {QWEN_SERVED_MODEL_NAME!r}"
    )

print("\n✅ TAAF/vLLM setup completed for Qwen3.8")
print("Model path:", QWEN_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


# atlas: remember what the bundle itself carries, before any override below.
ATLAS_PRISTINE = {
    "analyzer_timeout": getattr(bm.solver, "analyzer_timeout", None),
    "concurrency": getattr(bm.solver, "concurrency", None),
    "max_actions_per_game": getattr(bm.solver, "max_actions_per_game", None),
    "max_runtime_s_per_game": getattr(bm.solver, "max_runtime_s_per_game", None),
    "n_passes": getattr(bm, "n_passes", None),
}


In [ ]:
# Inline customization hook — Q38 P1-style public evaluation.
#
# Q38 P1 runs the full 25 public ARC-AGI-3 games once each (25 games × 1 pass).
# This override applies only to the public/offline notebook run. Competition reruns
# still replace bm.games from Kaggle's live gateway in the final run cell.

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))
print("Qwen3.8 model path:", os.environ.get("TAAF_QWEN_MODEL_PATH"))

Q38_P1_PUBLIC_GAME_IDS = [
    "ar25-0c556536",
    "bp35-0a0ad940",
    "cd82-fb555c5d",
    "cn04-2fe56bfb",
    "dc22-fdcac232",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ka59-38d34dbb",
    "lf52-271a04aa",
    "lp85-305b61c3",
    "ls20-9607627b",
    "m0r0-492f87ba",
    "r11l-495a7899",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "sc25-635fd71a",
    "sk48-d8078629",
    "sp80-589a99af",
    "su15-1944f8ab",
    "tn36-ef4dde99",
    "tr87-cd924810",
    "tu93-0768757b",
    "vc33-5430563c",
    "wa30-ee6fef47",
]

if not true_submission:
    if len(Q38_P1_PUBLIC_GAME_IDS) != 25 or len(set(Q38_P1_PUBLIC_GAME_IDS)) != 25:
        raise RuntimeError("Q38 P1 public game list must contain exactly 25 unique games.")
    if not bm.games:
        raise RuntimeError("benchmark_initial.pkl contains no template public game.")

    import taaf.game_api

    template_game = bm.games[0]
    arcade_spec = getattr(template_game, "arcade_spec", None)
    if arcade_spec is None:
        arcade_spec = getattr(template_game, "_arcade_spec", None)
    if arcade_spec is None:
        raise RuntimeError(
            "Could not recover the public ArcadeSpec from benchmark_initial.pkl; "
            "cannot construct the 25-game Q38 P1 evaluation set."
        )

    bm.games = [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=arcade_spec)
        for game_id in Q38_P1_PUBLIC_GAME_IDS
    ]
    bm.n_passes = 1
    bm.game_weights = None

    # These already match Q38 P1 in the source notebook; set them explicitly so the
    # intended evaluation configuration is visible and stable.
    if hasattr(bm.solver, "concurrency"):
        bm.solver.concurrency = 28
    if hasattr(bm.solver, "max_runtime_s_per_game"):
        bm.solver.max_runtime_s_per_game = 7920.0

    bm.label = f"{bm.label}-25g-p1"
    print(f"Public evaluation override: {len(bm.games)} games × {bm.n_passes} pass = {len(bm.games) * bm.n_passes} runs")
    print("Public evaluation concurrency:", getattr(bm.solver, "concurrency", None))
    print("Public per-game runtime cap (s):", getattr(bm.solver, "max_runtime_s_per_game", None))


In [ ]:
# ==========================================================================
# atlas v2 -- our layer on top of the upstream harness.
#
# v1 (22.08, analyzer_timeout 900 -> 180s, concurrency 28 -> 14) measured on
# arc3-atlas itself (22.08 night, 25 public games, 4h24m wall clock):
#
#   * mean score 4.99 (median 2.78), up from the 3.02 v0 baseline -- but:
#   * 512 analyzer requests failed, and 488 of those hit the FULL 180s cap
#     (vs. only 12/33 full-timeout stalls in the 900s baseline). That is
#     488 * 180s = 87840 thread-seconds of pure retry-thrash out of 221774
#     available (14 concurrency * 15841s wall clock) = 39.6% wasted.
#   * root cause, read from vllm-openai-server.log: aggregate generation
#     throughput holds at ~250-270 tok/s regardless of concurrency (GPU
#     compute-bound, KV cache usage only ~35-37%), so at 14 concurrent
#     requests each gets ~18-20 tok/s. LOCAL_ANALYZER_MAX_OUTPUT was 0
#     (uncompleted -- literally unbounded) with thinking enabled, so a
#     turn generating ~4800+ tokens already exceeds a 180s cap. 180s was a
#     guess, not measured -- it undershot badly.
#
# v2 fixes the actual cause instead of re-guessing the timeout in isolation:
#   1. Cap LOCAL_ANALYZER_MAX_OUTPUT at 8000 tokens. This is a MODULE-LEVEL
#      constant read once at import in inference.agent.tool_agent (line
#      ~148) and frozen again per-ToolAgent in __init__ -- setting the env
#      var here would be too late (the module is already imported by the
#      time this cell runs) and silently do nothing. Must patch the
#      module attribute directly so future ToolAgent() constructions (one
#      per game/pass, built fresh at run time -- NOT part of the pickle)
#      pick it up.
#   2. Raise analyzer_timeout to 480s: at the measured worst-case ~18 tok/s
#      (full 14-way contention), 8000 tokens takes ~444s: 480s gives ~35s
#      of margin for prefill/network on top of the now-bounded worst case.
#
# Score is driven by DEPTH, so time lost to queueing/retry-thrash is levels
# not reached.
# ==========================================================================

ATLAS_ANALYZER_TIMEOUT_S = 480.0       # v1 tried 180 (measured: far too short)
ATLAS_ANALYZER_MAX_OUTPUT_TOKENS = 8000  # v1 had this unbounded (0) -- the real bug
ATLAS_CONCURRENCY = 20  # substituted by the BUILDER (env ATLAS_CONCURRENCY_BUILD, default 20)
                                        # 29.08 evening (user, option B): BACK to
                                        # v23's proven waves-of-20. The one-wave
                                        # 110 config just scored 0.65 on the
                                        # hidden set with v23's own code (v24,
                                        # 55860197) vs 0.92 for waves-of-20 --
                                        # and the same-day testbed showed WHY:
                                        # 55 concurrent games starve each other
                                        # of LLM turns (wall time is cheap, LLM
                                        # turns are the scarce resource). This
                                        # build = round-6+8 depth code on the
                                        # config that actually scored 0.92.
                                        # Final-rescore (110 games -> 80 min/game)
                                        # config question deliberately deferred.
                                        # Previous one-wave rationale kept below
                                        # for history:
                                        # 28.08 (user): ONE WAVE of the ~110-game
                                        # hidden set (55 public-LB + 55 private,
                                        # structure confirmed 28.08). Rehearsals:
                                        # 55x4h and 110x6h both clean on the pod.
                                        # Was 20 through v23 (=6 waves, 80 min/game).
                                        # 27.08: kernel v21 Phase A calibration
                                        # (25 games, real Kaggle RTX Pro 6000
                                        # backend, not just RunPod A100) came
                                        # back clean at concurrency=20 -- no
                                        # retry-storm recurrence -- and the
                                        # user made the deliberate call to KEEP
                                        # 20 for the real (Phase B) submission
                                        # too. See [[arc-agi-3-top10-plan]]
                                        # memory for the full history.
                                        # 25.08: lowered from 14 after the
                                        # retry-storm bug (cn04/lp85/re86 in
                                        # v17 lost 15-30min each retrying one
                                        # analysis_step on request timeouts,
                                        # plausibly the shared local LLM
                                        # backend overloaded under 14x
                                        # concurrent load). Checked the real
                                        # atlas_fit_game_cap() formula first:
                                        # at 25 games, concurrency 9-13 all
                                        # still land on 3 (or fewer) waves,
                                        # so the real per-game cap stays
                                        # pinned at the 8500s ceiling either
                                        # way -- this costs ZERO per-game
                                        # budget, only raises the worst-case
                                        # total wall time (4.72h->7.08h,
                                        # still comfortably inside the 8h
                                        # budget/9h hard cap). Below 9,
                                        # waves=4 and the real cap drops to
                                        # 7200s -- a genuine cost, so 10 was
                                        # chosen for margin above that cliff
                                        # while cutting concurrent load by
                                        # ~29%. Complements, doesn't replace,
                                        # the retry-storm backstop in
                                        # solver.py -- less contention makes
                                        # a storm less likely, the backstop
                                        # bounds the damage if one still
                                        # happens. NOT applied to kernel
                                        # version 19 (already pushed/running
                                        # when this was decided) -- takes
                                        # effect on the next build after it.
                                        # See scripts/test_atlas_fit_game_cap.py
                                        # for the concurrency/waves/cap math.
ATLAS_FALLBACK_GAME_CAP_S = 7920.0     # applied only if the bundle carries none

# Wall-clock guard for the submission rerun. Unlike the offline run, a rerun
# gets soft_end_time=None, so the per-game cap is the ONLY thing standing
# between us and Kaggle killing the notebook at 9 h with no result.
ATLAS_SUBMISSION_BUDGET_S = 28800.0   # 29.08 (option B): back to v23's 8h +
                                       # 1h margin -- the exact budget behind
                                       # 0.92. Was 30600 (8.5h one-wave, v24's
                                       # 0.65). Original one-wave note:
                                       # 28.08 (user): 8.5 h, one-wave config --
                                       # no per-game extensions (draws off), so
                                       # this IS the per-game ceiling too. Rides
                                       # closer to the platform kill than the
                                       # old 8h+1h-margin: the 12h rule and
                                       # v23's ~9.6h elapsed rerun say it fits.
                                       # Margin left covers setup (dataset
                                       # mount, wheelhouse
                                       # install if not cached, vLLM start +
                                       # smoke test measured at ~5 min alone).
                                       # Raised from 7.5h/1.19-score run
                                       # 23.08 -- user's call, more play time
                                       # over more safety margin.
# 24.08: explicit, named ceiling for atlas_fit_game_cap() -- replaces reading
# bm.solver.max_runtime_s_per_game (7920s, the bundle's own undocumented
# default), which silently absorbed the whole budget increase above for any
# n_games <= 42 (v3/v5/v6/v8/v10/v12's real behavior). Raised 7920 -> 8500
# 28.08 (user's call: "так подними его"): raised 8500 -> 14400 (4h/game)
# alongside the planned Phase B concurrency increase. Context: the hidden
# Phase B set is ~110 games (55 semi-private + 55 fully-private per the
# ARC-AGI-3 technical report), so at concurrency 20 the affordable cap was
# only 80 min/game -- the leading explanation for v20's real 0.82 vs 1.43
# on the stand. With concurrency ~55, affordable becomes 14400s (110 games,
# 2 waves) or 28800s (55 games, 1 wave); this ceiling admits the 4h/game
# the 28.08 rehearsals validated while still bounding any single game to
# half the 8h budget. History: a v7 regression (0.06) came from removing
# the ceiling ENTIRELY -- keep it firm and named, never delete it.
ATLAS_SUBMISSION_GAME_CAP_CEILING_S = 8500.0  # 29.08 (option B): v23's ceiling
                                       # (was 30600 for one-wave). At 55 games /
                                       # 3 waves the affordable cap is 9600s ->
                                       # this ceiling binds at 8500s (~140 min),
                                       # the regime that scored 0.92.
ATLAS_MIN_GAME_CAP_S = 1800.0

print("atlas: solver config as it came from the bundle:")
for _key, _value in ATLAS_PRISTINE.items():
    print(f"atlas:   {_key} = {_value}")

bm.solver.analyzer_timeout = ATLAS_ANALYZER_TIMEOUT_S
bm.solver.concurrency = ATLAS_CONCURRENCY
if getattr(bm.solver, "max_runtime_s_per_game", None) is None:
    bm.solver.max_runtime_s_per_game = ATLAS_FALLBACK_GAME_CAP_S
    print(
        "atlas: bundle carried no per-game runtime cap; applied "
        f"{ATLAS_FALLBACK_GAME_CAP_S:.0f}s"
    )

import inference.agent.tool_agent as _atlas_tool_agent
_atlas_tool_agent._LOCAL_ANALYZER_MAX_OUTPUT = ATLAS_ANALYZER_MAX_OUTPUT_TOKENS
print(f"atlas: patched tool_agent._LOCAL_ANALYZER_MAX_OUTPUT = {ATLAS_ANALYZER_MAX_OUTPUT_TOKENS}")

# atlas 28.08 (one-wave config): module-attribute patches, NOT env vars --
# these constants are read at import, and the modules are already imported
# by the unpickle cell above, so os.environ would be silently too late
# (the exact trap the max-output comment above documents). Both patches are
# hasattr-guarded so this one notebook works with either dataset build
# (v23-agent hybrid or the full probes build).
import inference.framework.solver as _atlas_solver_mod
if hasattr(_atlas_solver_mod, "_ATLAS_TIME_BANK_DRAWS_ENABLED"):
    # 29.08 (option B): draws back ON -- v23's 0.92 regime included the time
    # bank, and at the 8500s ceiling a maxed-out draw (+100% => ~4.7h) stays
    # far under Kaggle's 12h kill. (The False patch was one-wave-specific:
    # a draw on top of an 8.5h cap would have blown the platform limit.)
    _atlas_solver_mod._ATLAS_TIME_BANK_DRAWS_ENABLED = True
    print("atlas: solver._ATLAS_TIME_BANK_DRAWS_ENABLED = True (v23 regime: bank draws allowed)")
if hasattr(_atlas_tool_agent, "_ATLAS_LLM_REQUEST_GATE"):
    import threading as _atlas_threading
    _atlas_tool_agent._ATLAS_LLM_MAX_CONCURRENT = 25
    _atlas_tool_agent._ATLAS_LLM_REQUEST_GATE = _atlas_threading.Semaphore(25)
    print("atlas: patched tool_agent LLM request gate = 25 (110 concurrent games)")
if hasattr(_atlas_tool_agent, "_ATLAS_LLM_ZOMBIE_GATE"):
    # 29.08 (round 6, D3): level-1 games with no progress get only 10 of
    # the 25 request slots -- level-2+ games keep priority (zombie cull).
    _atlas_tool_agent._ATLAS_LLM_ZOMBIE_SLOTS = 10
    _atlas_tool_agent._ATLAS_LLM_ZOMBIE_GATE = _atlas_threading.Semaphore(10)
    print("atlas: patched tool_agent zombie gate = 10 (level-1 no-progress cull)")

# Phase A ONLY: shrink the per-game cap for a quick calibration check of the
# timeout/max-output change above, on the real 25-game set (no fabricated
# repeats -- repeats of one game would raise vLLM's prefix-cache hit rate and
# understate contention). Same solver config, same real games, just cut
# short. Phase B (true_submission) is untouched -- atlas_fit_game_cap() below
# still sizes its cap from ATLAS_SUBMISSION_BUDGET_S alone.
# 29.08 FIX (caught live by the user via "Diff: +0 -0"): this constant
# lives INSIDE the generated notebook cell, so reading the env var here
# meant reading it AT KAGGLE RUNTIME (where it is unset) -- v25 silently
# ran a 4h Phase A instead of 30 min. The builder now substitutes the
# BUILD-time env value as a literal into the cell (see build()).
ATLAS_CALIBRATION_CAP_S = 7200.0  # value substituted by the BUILDER (env ATLAS_CALIBRATION_CAP_S at build time)
                                   # 28.08 (kernel v24, user): 4h/game --
                                   # Phase A becomes the MATCHED CONTROL
                                   # against the probe-branch decisive run
                                   # (25 public games, 4h each, one wave at
                                   # the new concurrency) on the kernel's
                                   # own RTX Pro 6000. ~4.7h of the 45h
                                   # weekly quota. Prior value 750s (v23's
                                   # 30-min total calibration):
                                   # user set a 30-min TOTAL budget (not
                                   # per-game -- see the wave math below).
                                   # 25 games / concurrency 20 = 2 waves;
                                   # 2*750s=1500s (25min) + ~5min setup ~=
                                   # 30min budget, same math as v21. This
                                   # push carries the planforce+rollbackfix
                                   # build (7265cbd: ATLAS_PLAN_FORCE_
                                   # OVERRIDE + the wa30 rollback gate-
                                   # bypass fix; deliberately WITHOUT the
                                   # try_actions/plan_real snapshot probes,
                                   # which stay RunPod-experiment-only for
                                   # now) -- the version the user will
                                   # submit for real. On RunPod A100 it
                                   # measured mean 0.96 / 7-of-25 games
                                   # with a level at the 57-min mark,
                                   # against 0.11 / 1-of-25 for the
                                   # previous (theoryforce) build at a
                                   # matched window.
if not true_submission:
    bm.solver.max_runtime_s_per_game = ATLAS_CALIBRATION_CAP_S
    print(f"atlas: Phase A calibration cap -- max_runtime_s_per_game = {ATLAS_CALIBRATION_CAP_S:.0f}s")


def atlas_fit_game_cap(n_games: int) -> None:
    """Shrink the per-game cap so every wave fits the notebook budget.

    Called from the submission branch, where the game list only becomes known
    after Kaggle's gateway answers.

    24.08: the ceiling is now the explicit ATLAS_SUBMISSION_GAME_CAP_CEILING_S
    constant, not whatever bm.solver.max_runtime_s_per_game happened to carry
    in from the bundle (7920s, undocumented) -- that implicit ceiling silently
    absorbed any budget increase for n_games<=42. This is still a firm
    ceiling (never removed outright, unlike the v7 regression) -- just a
    named, deliberately-chosen one instead of an inherited accident.
    """
    import json
    import math
    from datetime import datetime, timezone

    concurrency = max(1, int(bm.solver.concurrency))
    waves = max(1, math.ceil(max(1, int(n_games)) / concurrency))
    affordable = ATLAS_SUBMISSION_BUDGET_S / waves
    previous = float(bm.solver.max_runtime_s_per_game or ATLAS_FALLBACK_GAME_CAP_S)
    fitted = max(ATLAS_MIN_GAME_CAP_S, min(ATLAS_SUBMISSION_GAME_CAP_CEILING_S, affordable))
    print(
        f"atlas: {n_games} games / concurrency {concurrency} = {waves} wave(s); "
        f"per-game cap {previous:.0f}s -> {fitted:.0f}s (ceiling={ATLAS_SUBMISSION_GAME_CAP_CEILING_S:.0f}s)"
    )
    bm.solver.max_runtime_s_per_game = fitted

    # atlas: minimal_diagnostics=True on a real submission means the usual
    # summary.txt/transcripts never get written, and kernels output/logs may
    # not even reach a live competition rerun at all (unconfirmed -- the CLI
    # showed only the stale prior Phase A commit's files after 22.08's
    # submission finished). Write this anyway, on the chance it survives:
    # the one fact we actually want out of a real run is n_games itself.
    try:
        diagnostics_path = WORKING_DIR / "atlas_submission_diagnostics.json"
        diagnostics_path.write_text(
            json.dumps(
                {
                    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
                    "n_games": int(n_games),
                    "concurrency": concurrency,
                    "waves": waves,
                    "submission_budget_s": ATLAS_SUBMISSION_BUDGET_S,
                    "per_game_cap_before_s": previous,
                    "per_game_cap_after_s": fitted,
                },
                indent=2,
            ),
            encoding="utf-8",
        )
        print(f"atlas: wrote {diagnostics_path}")
    except Exception as exc:
        print(f"atlas: could not write submission diagnostics: {exc!r}")


print("atlas: effective solver config:")
print(f"atlas:   analyzer_timeout      = {bm.solver.analyzer_timeout}")
print(f"atlas:   analyzer_max_output   = {_atlas_tool_agent._LOCAL_ANALYZER_MAX_OUTPUT}")
print(f"atlas:   concurrency           = {bm.solver.concurrency}")
print(f"atlas:   max_runtime_s_per_game = {bm.solver.max_runtime_s_per_game}")
print(f"atlas:   max_actions_per_game  = {bm.solver.max_actions_per_game}")
print(f"atlas:   n_passes              = {bm.n_passes}")


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None
        # atlas 29.08 (option B): v23 regime -- time-bank draws ON (waves of
        # 20; deposits from stalled games get re-drawn by progressing ones,
        # ceiling 8500s keeps a maxed draw far under the platform kill).
        import os as _atlas_os
        _atlas_os.environ["ATLAS_TIME_BANK_DRAWS"] = "1"
        # atlas: a rerun has no soft deadline, so size the per-game cap to fit.
        atlas_fit_game_cap(len(bm.games))

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")